# 02 - Сommon Path Expressions (CPEs)

This notebook explores how **RTGL** handles more complex *multi-hop relationships* (where there is no single shortest temporal path between tables) within **Relational Deep Learning**. 

Unlike standard tasks, the user must explicitly define the required path via a **CPE** to connect tables.

The [**RelBench**](https://relbench.stanford.edu/) framework is used as the primary source of data and tasks, leveraging its collection of pre-defined tasks to evaluate **RTGL**'s capabilities.

## Table of Contents
1. [F1 Dataset](#f1-dataset)
    - 1.1 [Link Prediction Tasks](#f1-link-tasks)
        - 1.1.1 [driver-circuit-compete](#driver-circuit-compete)
2. [Stack-Exchange Q&A Website Dataset](#stack-exchange-dataset)
    - 2.1 [Link Prediction Tasks](#stack-link-tasks)
        - 2.1.1 [post-post-related](#post-post-related)
3. [Amazon e-commerce Dataset](#amazon-dataset)
    - 3.1 [Entity Regression Tasks](#amazon-reg-tasks)
        - 3.1.1 [item-ltv](#item-ltv)
4. [arXiv Dataset](#arxiv-dataset)
    - 4.1 [Entity Classification Tasks](#arxiv-clas-tasks)
        - 4.1.1 [paper-citation](#paper-citation)


In [26]:
%load_ext autoreload
%autoreload 2

from experiments.utils import load_dataset_rb, load_task_rb, check_correctness

## 1. F1 Dataset <a id="f1-dataset"></a>

In this section, we attempt to generate the same tasks from the `F1 Dataset` which are already pre-defined in *RelBench*.

In [27]:
dataset_f1 = load_dataset_rb(name="rel-f1")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 24 files:   0%|          | 0/24 [00:00<?, ?it/s]

### 1.1 Link Prediction Tasks <a id="f1-link-tasks"></a>

#### 1.1.1 driver-circuit-compete <a id="driver-circuit-compete"></a>

Task Description: Predict on which circuits a driver will compete in the next 1 year.

In [28]:
task_f1_driver_circuit = load_task_rb(dataset_f1, "driver-circuit-compete")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

In [29]:
rtgl_query = """
    WITH circuits_drivers AS (
        circuits.circuitId->races.circuitId:raceId->results.raceId:driverId->drivers.driverId
    )
    PREDICT LIST_DISTINCT(circuits_drivers.*, 0, 365, DAYS)
    FOR EACH drivers.*;
"""

In [30]:
# TRAIN

check_correctness(dataset_f1, task_f1_driver_circuit, rtgl_query, split="train")

TIMEDELTA: 365 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.07 seconds
------------------- START TRAIN -------------------
RelBench fkeys: {'driverId': 'drivers', 'circuitId': 'circuits'}
RelBench pkey: None
RelBench time col: date
RTGL fkeys: {'fk': 'drivers', 'label': 'circuits'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
       timestamp   fk                                              label _merge
0    2000-01-03    1  (0, 1, 3, 5, 6, 7, 8, 9, 10, 12, 13, 17, 18, 2...   both
1    2001-01-02    1  (0, 1, 3, 5, 6, 7, 8, 9, 10, 12, 13, 17, 18, 1...   both
2    2002-01-02    1  (0, 1, 3, 5, 6, 7, 8, 9, 10, 12, 13, 17, 18, 1...   both
3    2003-01-02    1  (0, 1, 3, 5, 6, 7, 8, 9, 10, 13, 17, 18, 19, 2...   both
4    2004-01-02    1  (0, 1, 2, 3, 5, 6, 7, 8, 9, 10, 12, 13, 16, 17...   both
...       

In [31]:
# VAL

check_correctness(dataset_f1, task_f1_driver_circuit, rtgl_query, split="val")

TIMEDELTA: 365 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.04 seconds
------------------- START VAL -------------------
RelBench fkeys: {'driverId': 'drivers', 'circuitId': 'circuits'}
RelBench pkey: None
RelBench time col: date
RTGL fkeys: {'fk': 'drivers', 'label': 'circuits'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
     timestamp  fk                                              label _merge
0  2005-01-01   1     (0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 18, 19, 20)   both
1  2005-01-01   3  (0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 16,...   both
2  2005-01-01   7  (0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 16,...   both
3  2005-01-01  10  (0, 2, 4, 6, 7, 8, 9, 10, 12, 13, 16, 17, 18, ...   both
4  2005-01-01  12  (0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 16,...   both
5  2005-01-01  13  (0, 1, 2, 3

In [32]:
# TEST

check_correctness(dataset_f1, task_f1_driver_circuit, rtgl_query, split="test")

TIMEDELTA: 365 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.03 seconds
------------------- START TEST -------------------
RelBench fkeys: {'driverId': 'drivers', 'circuitId': 'circuits'}
RelBench pkey: None
RelBench time col: date
RTGL fkeys: {'fk': 'drivers', 'label': 'circuits'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
     timestamp   fk                                              label _merge
0  2010-01-01    0  (0, 1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 13, 14...   both
1  2010-01-01    1                               (14, 17, 21, 23, 34)   both
2  2010-01-01    2  (0, 1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 13, 14...   both
3  2010-01-01    3  (0, 1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 13, 14...   both
4  2010-01-01    4  (0, 1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 13, 14...   both
5  2010-01-01    8  (0,

## 2. Stack-Exchange Q&A Website Dataset <a id="stack-exchange-dataset"></a>

In this section, we attempt to generate the same tasks from the `Stack-Exchange Q&A Website Dataset` which are already pre-defined in *RelBench*.


In [33]:
dataset_stack = load_dataset_rb(name="rel-stack")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

### 2.1 Link Prediction Tasks <a id="stack-link-tasks"></a>


#### 2.1.1 post-post-related <a id="post-post-related"></a>

Task Description: Predict a list of existing posts that users will link a given post to in the next two months.

In [34]:
task_stack_post_post = load_task_rb(dataset_stack, "post-post-related")

In [35]:
rtgl_query = """
    WITH links_posts AS (
        postLinks.PostId->posts.Id
    )
    PREDICT LIST_DISTINCT(links_posts.RelatedPostId, 0, 91, DAYS)
    FOR EACH posts.*;
"""

In [36]:
# TRAIN

check_correctness(dataset_stack, task_stack_post_post, rtgl_query, split="train")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.46 seconds
------------------- START TRAIN -------------------
RelBench fkeys: {'PostId': 'posts', 'postLinksIdList': 'posts'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'posts', 'label': 'posts'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
       timestamp      fk             label _merge
0    2012-01-12      28           (1427,)   both
1    2015-04-09      74           (4935,)   both
2    2019-10-03      74           (4935,)   both
3    2011-04-14      88           (1574,)   both
4    2016-01-07      90            (146,)   both
...         ...     ...               ...    ...
5850 2020-07-02  315039  (159209, 168163)   both
5851 2020-07-02  315054         (244965,)   both
5852 2020-07-02  315060          (98531,)   b

In [37]:
# VAL

check_correctness(dataset_stack, task_stack_post_post, rtgl_query, split="val")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.08 seconds
------------------- START VAL -------------------
RelBench fkeys: {'PostId': 'posts', 'postLinksIdList': 'posts'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'posts', 'label': 'posts'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
      timestamp      fk                     label _merge
0   2020-10-01     128                 (106097,)   both
1   2020-10-01     995                  (63486,)   both
2   2020-10-01    1574                     (88,)   both
3   2020-10-01    5030                  (20350,)   both
4   2020-10-01    8848                  (49884,)   both
..         ...     ...                       ...    ...
221 2020-10-01  324945                 (122765,)   both
222 2020-10-01  324967                  

In [38]:
# TEST

check_correctness(dataset_stack, task_stack_post_post, rtgl_query, split="test")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.09 seconds
------------------- START TEST -------------------
RelBench fkeys: {'PostId': 'posts', 'postLinksIdList': 'posts'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'posts', 'label': 'posts'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
      timestamp      fk                     label _merge
0   2021-01-01      43           (27860, 243921)   both
1   2021-01-01      51            (4010, 146052)   both
2   2021-01-01     210                 (328571,)   both
3   2021-01-01     342                   (7282,)   both
4   2021-01-01    1431                 (297150,)   both
..         ...     ...                       ...    ...
253 2021-01-01  333611          (114400, 189819)   both
254 2021-01-01  333637                 

## 3. Amazon e-commerce Dataset <a id="amazon-dataset"></a>

In this section, we attempt to generate the same tasks from the `Amazon e-commerce Dataset` which are already pre-defined in *RelBench*.


In [39]:
dataset_amazon = load_dataset_rb(name="rel-amazon")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 34 files:   0%|          | 0/34 [00:00<?, ?it/s]

### 3.1 Entity Regression Tasks <a id="amazon-reg-tasks"></a>


#### 3.1.1 item-ltv <a id="item-ltv"></a>

Task Description: For each product, predict the $ value of the total number purchases and reviews it recieves in the next 3 months.

In [40]:
task_amazon_item_ltv = load_task_rb(dataset_amazon, "item-ltv")

In [41]:
rtgl_query = """
     WITH product_product AS (
          product.product_id->review.product_id->product.product_id
     )
     PREDICT SUM(product_product.price, 0, 91, DAYS)
     FOR EACH product.*
     ASSUMING COUNT(review.*, 0, 91, DAYS) != 0;
"""

In [42]:
# TRAIN

check_correctness(dataset_amazon, task_amazon_item_ltv, rtgl_query, split="train")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

SQL query executed in 9.15 seconds
------------------- START TRAIN -------------------
RelBench fkeys: {'product_id': 'product'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'product'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
          timestamp      fk    label _merge
0       2012-10-04       0   899.99   both
1       2013-01-03       0  2197.65   both
2       2013-04-04       0  1737.19   both
3       2013-07-04       0  2030.21   both
4       2013-10-03       0  1862.77   both
...            ...     ...      ...    ...
2707674 2013-10-03  506009    85.86   both
2707675 2014-07-03  506009   171.72   both
2707676 2014-10-02  506009   171.72   both
2707677 2008-10-09  506010    17.87   both
2707678 2012-10-04  506010    17.87   both

[2707679 rows x 4 columns]
------------------- END TRAIN -

In [43]:
# VAL

check_correctness(dataset_amazon, task_amazon_item_ltv, rtgl_query, split="val")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.36 seconds
------------------- START VAL -------------------
RelBench fkeys: {'product_id': 'product'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'product'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
         timestamp      fk    label _merge
0      2015-10-01       0  2344.16   both
1      2015-10-01       3   159.95   both
2      2015-10-01       4   358.08   both
3      2015-10-01       5     3.99   both
4      2015-10-01       6    47.70   both
...           ...     ...      ...    ...
166973 2015-10-01  505996    21.35   both
166974 2015-10-01  505998    17.69   both
166975 2015-10-01  505999   111.18   both
166976 2015-10-01  506000  1277.92   both
166977 2015-10-01  506002    19.82   both

[166978 rows x 4 colu

In [44]:
# TEST

check_correctness(dataset_amazon, task_amazon_item_ltv, rtgl_query, split="test")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.41 seconds
------------------- START TEST -------------------
RelBench fkeys: {'product_id': 'product'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'product'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
         timestamp      fk    label _merge
0      2016-01-01       0  2448.81   both
1      2016-01-01       1    23.98   both
2      2016-01-01       4   134.28   both
3      2016-01-01       6    23.85   both
4      2016-01-01       7     5.78   both
...           ...     ...      ...    ...
178329 2016-01-01  505998    35.38   both
178330 2016-01-01  505999    74.12   both
178331 2016-01-01  506000  1069.28   both
178332 2016-01-01  506002    19.82   both
178333 2016-01-01  506008    17.00   both

[178334 rows x 4 col

## 4. arXiv Dataset <a id="arxiv-dataset"></a>

In this section, we attempt to generate the same tasks from the `arXiv Dataset` which are already pre-defined in *RelBench*.


In [45]:
dataset_arxiv = load_dataset_rb(name="rel-arxiv")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 25 files:   0%|          | 0/25 [00:00<?, ?it/s]

### 4.1 Entity Classification Tasks <a id="arxiv-clas-tasks"></a>


#### 4.1.1 paper-citation <a id="paper-citation"></a>

Task Description: Predict if a paper gets cited in the next 6 months.

In [46]:
task_arxiv_paper_citation = load_task_rb(dataset_arxiv, "paper-citation")

In [47]:
rtgl_query = """
    WITH citations_papers AS (
        citations.References_Paper_ID->papers.Paper_ID
    )
    PREDICT COUNT(citations_papers.*, 0, 182, DAYS) != 0
    FOR EACH papers.*;
"""

In [48]:
# TRAIN

check_correctness(dataset_arxiv, task_arxiv_paper_citation, rtgl_query, split="train")

TIMEDELTA: 182 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.36 seconds
------------------- START TRAIN -------------------
RelBench fkeys: {'Paper_ID': 'papers'}
RelBench pkey: None
RelBench time col: date
RTGL fkeys: {'fk': 'papers'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
         timestamp      fk label _merge
0      2018-01-06       0     0   both
1      2018-07-07       0     0   both
2      2019-01-05       0     0   both
3      2019-07-06       0     0   both
4      2020-01-04       0     0   both
...           ...     ...   ...    ...
534228 2021-07-03  136178     0   both
534229 2021-07-03  136179     0   both
534230 2021-07-03  136180     0   both
534231 2021-07-03  136181     0   both
534232 2021-07-03  136182     0   both

[534233 rows x 4 columns]
------------------- END TRAIN -------

In [49]:
# VAL

check_correctness(dataset_arxiv, task_arxiv_paper_citation, rtgl_query, split="val")

TIMEDELTA: 182 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.12 seconds
------------------- START VAL -------------------
RelBench fkeys: {'Paper_ID': 'papers'}
RelBench pkey: None
RelBench time col: date
RTGL fkeys: {'fk': 'papers'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
         timestamp      fk label _merge
0      2022-01-01       0     0   both
1      2022-01-01       1     0   both
2      2022-01-01       2     1   both
3      2022-01-01       3     0   both
4      2022-01-01       4     0   both
...           ...     ...   ...    ...
155840 2022-01-01  155840     0   both
155841 2022-01-01  155841     0   both
155842 2022-01-01  155842     0   both
155843 2022-01-01  155843     0   both
155844 2022-01-01  155844     1   both

[155845 rows x 4 columns]
------------------- END VAL -----------

In [50]:
# TEST

check_correctness(dataset_arxiv, task_arxiv_paper_citation, rtgl_query, split="test")

TIMEDELTA: 182 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.13 seconds
------------------- START TEST -------------------
RelBench fkeys: {'Paper_ID': 'papers'}
RelBench pkey: None
RelBench time col: date
RTGL fkeys: {'fk': 'papers'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
         timestamp      fk label _merge
0      2023-01-01       0     0   both
1      2023-01-01       1     0   both
2      2023-01-01       2     1   both
3      2023-01-01       3     0   both
4      2023-01-01       4     0   both
...           ...     ...   ...    ...
193691 2023-01-01  193691     0   both
193692 2023-01-01  193692     0   both
193693 2023-01-01  193693     0   both
193694 2023-01-01  193694     0   both
193695 2023-01-01  193695     0   both

[193696 rows x 4 columns]
------------------- END TEST ---------